This notebook uses the following inputs:
1. Harmonised socioeconomic indicators for the base year, saved as a geopackage file (output from step 1: **baseyear_prep**)
2. Historical national Gini indices from the World Bank, saved as an Excel workbook
3. Global base year socioeconomic and electricity-related indicators, saved as an Excel workbook 

in order to:
1. Re-distribute base year income data using Acklam's approximation for parameterised lognormal distributions^
2. Estimate base year residential electricity demand

^Base year income is re-distributed since the income distribution from Kummu et al. (v4) varies significantly from reported figures by the World Bank.

Base year results are validated against:
1. Reported income percentiles from the World Bank, saved as an Excel workbook
2. Reported residential electricity demand from the IEA, saved as an excel workbook

This notebook is only used for historical validation purposes and does not feed into any other scripts used for future projections.

## 1. Importing required packages

The following cell needs to be run first whenever the kernel is restarted.

In [ ]:
from geocube.vector import vectorize

import geopandas as gpd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import numpy as np

import pandas as pd

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import show
from rasterio.features import rasterize
from rasterio.transform import from_origin

import rioxarray

from rtree import index

from scipy.optimize import curve_fit
from scipy.special import erf, erfinv
from scipy.stats import lognorm

from shapely.ops import nearest_points
from shapely.geometry import box

from sklearn.metrics import r2_score

import mapclassify

## 2. Country list and input files

In [ ]:
# User-defined country list, based on upper-case 3-letter country code
ccode_list = ['NAM']

In [ ]:
# Import global socioeconomic indicators
# National gini indices in 2015, from the World Bank
gini_wb = pd.read_excel('input/WB_Gini_2015.xlsx', sheet_name='Gini_2015')

# National income percentiles in 2015, from the World Bank
incdist_wb = pd.read_excel('input/WB_INC_DIST_2015.xlsx', sheet_name='INC_DIST_2015')

# National socioeconomic and electricity-related indicators in 2015, from the SSP, World Bank and IEA
# incl. pop, gdp, inc, elec access, residential elec demand
socioecon_base = pd.read_excel('input/socioeconomic_data_ssp_iea_2015.xlsx', sheet_name='Data')

In [ ]:
# Import harmonised base year socioeconomic data for all countries under investigation
# (output of baseyear_prep)
for ccode in ccode_list:
    base_name = ccode + '_base'
    gdp_name = ccode + '_gdp'
    pop_name = ccode + '_pop'
    
    locals()[base_name] = gpd.read_file('1_output/' + ccode + '_socioecon_base_2015_Kummu.gpkg')

    # calculate normalised population (i.e. pop share) for each cell
    locals()[base_name]['Norm_POP'] = locals()[base_name]['2015_POP'] / (locals()[base_name]['2015_POP'].sum() + 0.000001)
    
    # store original total GDP and population count
    locals()[gdp_name] = locals()[base_name]['2015_GDP_corr'].sum()
    locals()[pop_name] = locals()[base_name]['2015_POP'].sum()

# display one of the resulting geodataframes    
NAM_base.head()

## 3. Income distribution

### 3.1 Initial verification

The purpose of this step is to verify the Gini index of the GDP data obtained from Kummu et al. (v4) against reported values obtained from the World Bank.

In [ ]:
for ccode in ccode_list:
    base_name = ccode + '_base'
    sorted_name = ccode + '_sorted'
    
    # sort cells ascendingly according to their income level
    locals()[sorted_name] = locals()[base_name].sort_values(by=['2015_INC_corr'])
    
    # calculate the cumulative share of population
    locals()[sorted_name]['POP_cshare'] = locals()[sorted_name]['2015_POP'].cumsum() / (locals()[sorted_name]['2015_POP'].sum() + 0.000001)
    
    # calculate the cumulative share of GDP
    locals()[sorted_name]['GDP_cshare'] = locals()[sorted_name]['2015_GDP_corr'].cumsum() / locals()[sorted_name]['2015_GDP_corr'].sum()

# display dataset
NAM_sorted

In [ ]:
# create empty list to compare calculated and reported gini indices
gini_results = []

for ccode in ccode_list:
    sorted_name = ccode + '_sorted'
    
    # calculate area under the Lorenz curve of cumulative population vs. cumulative GDP (based on Kummu at el.)
    auc_calc = np.trapz(locals()[sorted_name]['GDP_cshare'], x=locals()[sorted_name]['POP_cshare'])
    
    # calculate the gini index (based on Kummu et al.)
    gini_calc = (0.5 - auc_calc) / 0.5
    
    # retrieve official gini statistics from World Bank
    gini_org = gini_wb.loc[gini_wb['Country Code'] == ccode, 'Gini_2015'].values[0]
    # append country results to list
    gini_results.append({'Country': ccode, 'Gini_calculated': f'{gini_calc:,.3f}', 'Gini_WB': f'{gini_org:,.3f}'})

# create a dataframe based on the full list
gini_results_df = pd.DataFrame(gini_results)

# display results
gini_results_df

### 3.2 Income re-distribution

The purpose of this step is to re-distribute income so that the resulting Gini index matches the reported one from the World Bank. This is done since the calculated Gini index based on GDP data from Kummu et al. (v4) varies significantly from reported figures.

Income is re-distributed according to a lognormal distribution parameterised by mean income and Gini index.
This is realised via Acklam's rational approximation of lognormal distributions, which ensures internal consistency of mean income, Gini index, and total GDP.

In [ ]:
### LOG-NORMAL INCOME DISTRIBUTION USING ACKLAM'S APPROXIMATION ###

# ------------------------ Normal CDF & inverse CDF ------------------------ #

def norm_cdf(x: np.ndarray) -> np.ndarray:
    """Standard normal CDF Φ(x) using erf."""
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

def norm_ppf(p: np.ndarray) -> np.ndarray:
    """
    Inverse standard normal CDF (Acklam’s approximation).
    Accurate to ~1e-9 over (0,1).
    """
    p = np.asarray(p, dtype=float)
    if np.any((p <= 0) | (p >= 1)):
        raise ValueError("All probabilities must lie strictly between 0 and 1 (exclusive).")

    a = np.array([-3.969683028665376e+01,  2.209460984245205e+02,
                  -2.759285104469687e+02,  1.383577518672690e+02,
                  -3.066479806614716e+01,  2.506628277459239e+00])
    b = np.array([-5.447609879822406e+01,  1.615858368580409e+02,
                  -1.556989798598866e+02,  6.680131188771972e+01,
                  -1.328068155288572e+01])
    c = np.array([-7.784894002430293e-03, -3.223964580411365e-01,
                  -2.400758277161838e+00, -2.549732539343734e+00,
                   4.374664141464968e+00,  2.938163982698783e+00])
    d = np.array([ 7.784695709041462e-03,  3.224671290700398e-01,
                   2.445134137142996e+00,  3.754408661907416e+00])

    plow, phigh = 0.02425, 1 - 0.02425
    x = np.empty_like(p)

    # Lower region
    mask = p < plow
    if np.any(mask):
        q = np.sqrt(-2 * np.log(p[mask]))
        x[mask] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                   ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)

    # Central region
    mask = (p >= plow) & (p <= phigh)
    if np.any(mask):
        q = p[mask] - 0.5
        r = q*q
        x[mask] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
                   (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

    # Upper region
    mask = p > phigh
    if np.any(mask):
        q = np.sqrt(-2 * np.log(1 - p[mask]))
        x[mask] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                    ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    return x

# ------------------------ Lognormal parameter mapping ------------------------ #

def lognormal_params_from_mean_gini(mean_income: float, gini_index: float) -> tuple[float, float]:
    """
    Map arithmetic mean and Gini to (mu, sigma) for a Lognormal.
      mean = exp(mu + 0.5*sigma^2)
      Gini = 2*Phi(sigma/sqrt(2)) - 1
    """
    if mean_income <= 0:
        raise ValueError("mean_income must be positive.")
    if not (0 < gini_index < 1):
        raise ValueError("gini_index must be in (0,1).")

    p = (gini_index + 1.0) / 2.0
    sigma = np.sqrt(2.0) * norm_ppf(np.array([p]))[0]
    mu = np.log(mean_income) - 0.5 * sigma**2
    return mu, sigma

# ------------------------ Quantiles and bin means from cumulative shares ------------------------ #

def incomes_at_cum_shares(mean_income: float,
                          gini_index: float,
                          cum_shares: np.ndarray,
                          *,
                          mode: str = "quantile") -> np.ndarray:
    """
    Return income levels corresponding to cumulative population shares.

    Parameters
    ----------
    mean_income : float
        Target arithmetic mean of the distribution.
    gini_index : float
        Target Gini coefficient (0 < G < 1).
    cum_shares : array_like
        Monotonic increasing cumulative population shares in (0,1]; e.g., [0.1, 0.2, ..., 1.0].
    mode : {'quantile', 'bin_mean'}
        'quantile' : returns Q(p_i) = exp(mu + sigma * Phi^{-1}(p_i)) at each cumulative share p_i.
        'bin_mean' : returns average income within each interval (p_{i-1}, p_i]; length equals cum_shares.size.

    Returns
    -------
    incomes : np.ndarray
        Income at each cumulative share (quantile), or average income within each cumulative bin (bin_mean).

    Notes
    -----
    - 'quantile' gives income thresholds; useful for Lorenz curves and percentiles.
    - 'bin_mean' gives income **averages per group**; if you weight each bin by its share width and multiply by population,
      total GDP equals mean_income × population_size (up to floating error).
    """
    cum_shares = np.asarray(cum_shares, dtype=float)
    if np.any((cum_shares <= 0) | (cum_shares > 1)) or np.any(np.diff(cum_shares) <= 0):
        raise ValueError("cum_shares must be strictly increasing and lie in (0,1], e.g., [0.1, 0.2, ..., 1.0].")

    mu, sigma = lognormal_params_from_mean_gini(mean_income, gini_index)

    if mode == "quantile":
        # Avoid endpoints that cause +/-inf in the inverse CDF
        eps = 1e-12
        p = np.clip(cum_shares, eps, 1 - eps)
        z = norm_ppf(p)
        incomes = np.exp(mu + sigma * z)
        return incomes

    elif mode == "bin_mean":
        # Compute mean income within each bin (p_{i-1}, p_i], using truncated lognormal moments.
        # Let Y ~ N(mu, sigma^2), X = exp(Y). For bounds a,b in Y-space:
        #   E[X | a<Y<b] = exp(mu + 0.5*sigma^2) *
        #                  [Phi((b - mu - sigma^2)/sigma) - Phi((a - mu - sigma^2)/sigma)] /
        #                  [Phi((b - mu)/sigma)           - Phi((a - mu)/sigma)]
        p_edges = np.concatenate(([0.0], cum_shares))           # include 0 as left edge
        # Map edges in probability space to Y-space bounds (allow +/-inf at 0 and 1)
        y_edges = np.empty_like(p_edges)
        mask0 = p_edges == 0.0
        mask1 = p_edges == 1.0
        maskm = (~mask0) & (~mask1)
        y_edges[mask0] = -np.inf
        y_edges[mask1] = +np.inf
        if np.any(maskm):
            y_edges[maskm] = mu + sigma * norm_ppf(p_edges[maskm])

        yL = y_edges[:-1]
        yU = y_edges[1:]

        # Denominator P(a<Y<b)
        denom = norm_cdf((yU - mu) / sigma) - norm_cdf((yL - mu) / sigma)
        # Numerator piece for E[exp(Y) 1{a<Y<b}]
        numer = norm_cdf((yU - mu - sigma**2) / sigma) - norm_cdf((yL - mu - sigma**2) / sigma)

        # Handle any tiny denominators robustly
        small = denom < 1e-15
        with np.errstate(divide='ignore', invalid='ignore'):
            bin_means = np.exp(mu + 0.5 * sigma**2) * (numer / denom)
        # If a bin is extremely small, fallback to midpoint approximation
        if np.any(small):
            # Use the quantile at bin mid-point
            p_mid = 0.5 * (p_edges[:-1] + p_edges[1:])
            z_mid = norm_ppf(np.clip(p_mid, 1e-12, 1-1e-12))
            bin_means[small] = np.exp(mu + sigma * z_mid[small])

        return bin_means

    else:
        raise ValueError("mode must be 'quantile' or 'bin_mean'.")


In [ ]:
# Re-distributing base year income levels for all countries under investigation

for ccode in ccode_list:
    sorted_name = ccode + '_sorted'
    gini_name = ccode + '_gini'
    gdp_name = ccode + '_gdp'
    pop_name = ccode + '_pop'
    inc_name = ccode + '_inc'

    # retrieve gini and mean income, used to parameterise the income distribution
    locals()[gini_name] = gini_wb.loc[gini_wb['Country Code'] == ccode, 'Gini_2015'].values[0]
    locals()[inc_name] = locals()[gdp_name] / locals()[pop_name]

    # create new dataframe to store re-distributed income
    redist_name = ccode + '_redist_inc'
    locals()[redist_name] = locals()[sorted_name].copy()
    
    locals()[redist_name]['gini'] = locals()[gini_name] # add column with base year Gini index
    locals()[redist_name]['mean_inc'] = locals()[inc_name] # add column with base year mean income

    # re-distribute income using acklam's approximation
    cum_shares = locals()[redist_name]['POP_cshare']
    
    locals()[redist_name]['INC_p'] = incomes_at_cum_shares(locals()[inc_name], locals()[gini_name], cum_shares, mode="bin_mean")

    # calculate new GDP per cell
    locals()[redist_name]['GDP_p'] = locals()[redist_name]['INC_p'] * locals()[redist_name]['2015_POP']

    # verify resulting total GDP
    print('Results for', ccode)
    print('gini =', locals()[gini_name])
    print('total GDP, after re-distribution =', f'{locals()[redist_name]["GDP_p"].sum():,.0f}')
    print('total GDP, original =', f'{locals()[gdp_name]:,.0f}')
    print('relative error =', f'{(locals()[redist_name]["GDP_p"].sum() - locals()[gdp_name]) / locals()[gdp_name]:,.2%}')

    # calculate new GDP share
    locals()[redist_name]['GDP_share_p'] = locals()[redist_name]['GDP_p'] / locals()[redist_name]['GDP_p'].sum()
    
    # calculate new cumulative GDP share
    locals()[redist_name]['GDP_cshare_p'] = locals()[redist_name]['GDP_share_p'].cumsum()

    # drop unnecessary columns
    locals()[redist_name] = locals()[redist_name].drop(columns=['2015_GDP_corr', '2015_INC_corr', 'GDP_cshare'])
    

In [ ]:
# display one of the resulting GeoDataFrames
NAM_redist_inc

### 3.3 Validation of income re-distribution method

The purpose of this step is to validate the income distribution resulting from using Acklam's algorithm against reported values from the World Bank. The World Bank reports the following indicators, which are used to construct the income distribution Lorenz curve:
- Income share held by lowest 10%
- Income share held by lowest 20%
- Income share held by second 20%
- Income share held by third 20%
- Income share held by fourth 20%
- Income share held by highest 20%
- Income share held by highest 10%

The high-resolution GeoDataFrame resulting from Acklam's algorithm produces a more continuous Lorenz curve. The percentile values corresponding to the ones available from the World Bank are used to compute the root mean square error, as a measure of accuracy.

In [ ]:
# retrieve reported income shares from the World Bank
for ccode in ccode_list:
    incdist_name = ccode + '_incdist' # cumulative shares
    incdist_int = ccode + '_incdist_int' # interval shares instead of cumulative shares
    
    locals()[incdist_name] = [0]
    locals()[incdist_int] = []
    
    percentiles = [10, 20, 40, 60, 80, 90, 100] # percentiles reported by the World Bank

    for i, p in enumerate(percentiles):
        # retrieve cumulative income shares
        locals()[incdist_name].append(incdist_wb.loc[incdist_wb['Country Code'] == ccode, 'P'+str(p)].values[0] / 100)
        
        # retrieve interval income shares
        if i == 0:
            locals()[incdist_int].append(incdist_wb.loc[incdist_wb['Country Code'] == ccode, 'P0-'+str(p)].values[0] / 100)
        else:
            locals()[incdist_int].append(incdist_wb.loc[incdist_wb['Country Code'] == ccode, 'P'+str(percentiles[i-1])+'-'+str(p)].values[0] / 100)
        

# display one of the resulting lists        
NAM_incdist_int

In [ ]:
# retrieve the corresponding income shares predicted by Acklam's algorithm to calculate RMSE
for ccode in ccode_list:
    redist_name = ccode + '_redist_inc'
    incdist_pred_name = ccode + '_incdist_pred' # cumulative shares
    incdist_pred_int = ccode + '_incdist_pred_int' # interval shares instead of cumulative shares
    rmse_name = ccode + '_rmse'

    locals()[incdist_pred_name] = [0]

    percentiles = [10, 20, 40, 60, 80, 90] # percentiles reported by the World Bank

    # retrieving predicted cumulative shares 
    for p in percentiles:
        df = locals()[redist_name].copy()
        # difference between target percentile and cumulative population share of each cell
        df['POP_diff'] = (p/100 - df['POP_cshare']).abs()
        # retrieving index of the cell with minimum difference (closest match)
        pop_idx = df['POP_diff'].idxmin()
        # retrieving corresponding income share of the identified cell
        locals()[incdist_pred_name].append(df.loc[pop_idx, 'GDP_cshare_p'])

    locals()[incdist_pred_name].append(1)

    # calculating predicted interval shares
    p_list = locals()[incdist_pred_name].copy()
    locals()[incdist_pred_int] = [p if i==0 else p - (p_list[i-1]) for i, p in enumerate(p_list)]

    # calculating RMSE
    meanSquaredError = ((np.array(locals()[incdist_pred_int][1:]) - np.array(locals()[incdist_int])) ** 2).mean()
    locals()[rmse_name] = np.sqrt(meanSquaredError)
    print('RMSE,', ccode, '=', f'{locals()[rmse_name]:.3f}')


In [ ]:
# plotting income Lorenz curves based on official values from the World Bank and predicted values by Acklam's algorithm
for ccode in ccode_list:
    incdist_name = ccode + '_incdist' # World Bank
    redist_name = ccode + '_redist_inc' # Acklam's algorithm
    rmse_name = ccode + '_rmse'

    # plot reported income distribution based on values retrieved from the World Bank
    dec_pop_wb = [0, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1] # x-axis values
    plt.plot(dec_pop_wb, locals()[incdist_name], color='black', label='Reported, World Bank')
    
    # plot reproduced income distribution using Lognormal model via Acklam's algorithm
    plt.plot(locals()[redist_name]['POP_cshare'], locals()[redist_name]['GDP_cshare_p'], color='red', linestyle='--',
             label=f'Estimated, Lognormal model \nRMSE = {locals()[rmse_name]:.3f}')
    
    # plot line of perfect equality
    plt.plot([0, 1], [0, 1], color='gray', linestyle='dotted')

    # plot properties, change as needed
    plt.xlabel('Cumulative population share [-]')
    plt.ylabel('Cumulative GDP share [-]')
    plt.grid(which='both', alpha=0.25)
    plt.rcParams['font.family'] = 'sans-serif' # font type
    plt.rcParams['font.size'] = 7 # font size
    plt.legend()
    plt.savefig('2_output/INC_Dist_validation.png', dpi=300) # saving the generated plot: file name and resolution
    plt.show()

## 4. Residential electricity demand

The purpose of this step is to estimate residential electricity demand based on the socioeconomic indicators of each individual cell, which can then be aggregated into larger regions, or the whole country. The estimation is done based on a sigmoid function (S-curve) that correlates residential electricity demand to income level. The S-curve is fitted based on global income and electricity demand statistics, as follows:
- Country-level mean income level for the base year obtained from the SSP historical reference values, saved as an Excel workbook
- Country-level residential electricity demand for the base year obtained from the IEA, saved as an Excel workbook

After estimation and aggregation, the resulting total electricity demand is compared to the reported value from the IEA for validation.

### 4.1 S-curve fitting

In [ ]:
# Correlation between electricity demand and income level based on global statistics using a sigmoid function
# defining a sigmoid function S(x) = min + (max-min) * {(1 /(1+exp(-k(x-x0)))^a}
min_e = np.log10(0.9) # min electricity demand = 0.9 kWh/capita/year, equivalent to MTF tier 1
max_e = np.log10(7500) # max electricity demand = 7,500 kWh/capita/year, equivalent to highest global figure (Norway)

def sigmoid(x, k, a, x0):
    return 10 ** (min_e + (max_e - min_e) * (1/(1 + np.exp(-k * (np.log10(x)-np.log10(x0))))**a))

# providing an initial guess for the parameters (coefficients)
initial_guess = [1.5, 0.5, 26000]

# fitting the two-term exponential model
x_data = socioecon_base['INC_2017USD_calc']
y_data = socioecon_base['Res_Elec_kWhCapita']
params, _ = curve_fit(sigmoid, x_data, y_data, p0=initial_guess, maxfev=100000)

# displaying the sigmoid equation coefficients
k, a, x0 = params
print(f"x0 = {x0:,.0f}, k = {k:.1f}, a = {a:.1f}")

# calculating the predicted values
y_pred = sigmoid(x_data, *params)

# calculating R-squared value, as an indicator for model fitness
r_squared = r2_score(y_data, y_pred)
print(f"R-squared: {r_squared:.2f}")

# plotting the data and the two-term exponential fit on a logarithmic scale
fig, ax = plt.subplots()
plt.scatter(x_data, y_data, label='Data', s=100, c='yellow') # original data points

# adding labels to data points
for i, txt in enumerate(socioecon_base['Code']):
    ax.annotate(txt, (x_data.iat[i], y_data.iat[i]), fontsize=6, alpha=1, ha='center', va='center')

# s-curve trendline
x_fit = np.linspace(100, 150000, 10000)
y_fit = sigmoid(x_fit, *params)

# plotting trendline and legend
plt.plot(x_fit, y_fit, color='red',
         label=f"$x_{0}$ = {x0:,.0f}, k = {k:.1f}, a = {a:.1f}\n$R^{2}$ value = {r_squared:.2f}")
plt.xscale('log') # x-axis scale
plt.yscale('log') # y-axis scale
plt.xlabel('Income level, PPP [const. intl. 2017 USD/cap]') # x-axis label
plt.ylabel('Residential electricity demand [kWh/cap]') # y-axis label
plt.ylim(top=10000) # upper limit of y-axis range
plt.legend() # display legend
plt.grid(which='both', alpha=0.25) # show gridlines
plt.rcParams['font.family'] = 'sans-serif' # font type
plt.rcParams['font.size'] = 7 # font size
plt.savefig('2_output/INC_ResElec_corr_sigmoid.png', dpi=300) # saving the generated plot: file name and resolution
plt.show()

# displaying GDP and electricity consumption stats table for reference
socioecon_base.head()

### 4.2 Demand estimation and MTF classification

In [ ]:
# Multi-Tier Framework classification
mtf_tiers_hh = [4.5, 73, 365, 1250, 3000] # in kWh/hh/year, annual household consumption, 5-person household (reference)
hh_size = 5 # reference household

# calculating equivalent per capita consumption based on household size
mtf_tiers_cap = [tier / hh_size for tier in mtf_tiers_hh] # in kWh/cap/year, annual per capita consumption

# printing resulting MTF classes for the study region, change labels as needed
print(f'Multi-Tier Framework classification based on reference household size:\nTier 1 >= {mtf_tiers_cap[0]:.1f} kWh/cap/year\nTier 2 >= {mtf_tiers_cap[1]:.0f} kWh/cap/year\nTier 3 >= {mtf_tiers_cap[2]:.0f} kWh/cap/year\nTier 4 >= {mtf_tiers_cap[3]:.0f} kWh/cap/year\nTier 5 >= {mtf_tiers_cap[4]:.0f} kWh/cap/year')

# Min. electricity demand to achieve universal electricity access per MTF definition
min_elec = mtf_tiers_cap[0]

# Max. electricity demand capped at highest global value (Norway)
max_elec = 7500 
# Max. electricity demand capped at current national average for currently unelectrified cells
#max_unelec = avg_res_elec

# creating bins and labels for each Tier for later electricity consumption classification
mtf_bins = mtf_tiers_cap.copy()
mtf_bins.append(max_elec+1)
labels = [1, 2, 3, 4, 5]

In [ ]:
# residential electricity demand estimation using fitted S-curve
for ccode in ccode_list:
    redist_name = ccode + '_redist_inc'
            
    # estimating per capita electricity demand based on income level using sigmoid function
    locals()[redist_name]['2015_perCapitaElec'] = sigmoid(locals()[redist_name]['INC_p'], *params)

    # calculating total demand per cell based on population count
    locals()[redist_name]['2015_TotElec'] = locals()[redist_name]['2015_perCapitaElec'] * locals()[redist_name]['2015_POP']
            
    # classifying each cell according to MTF based on per capita demand level
    locals()[redist_name]['2015_MTF'] = pd.cut(locals()[redist_name]['2015_perCapitaElec'], bins=mtf_bins, labels=labels, right=False)

# displaying the resulting GeoDataFrame
NAM_redist_inc.head()

In [ ]:
NAM_redist_inc['2015_MTF'] = np.where(NAM_redist_inc['elec_stat'] == False, 0, NAM_redist_inc['2015_MTF'])
NAM_redist_inc

### 4.3 Validation of demand estimation method

In [ ]:
for ccode in ccode_list:
    redist_name = ccode + '_redist_inc'
    
    # calculating country-level residential electricity demand, only for electrified cells
    reselec_est = locals()[redist_name][locals()[redist_name]['elec_stat']==True]['2015_TotElec'].sum() / 10**9 # TWh
    # retrieving reported value from the IEA
    reselec_iea = socioecon_base.loc[socioecon_base['Code'] == ccode.lower(), 'Res_Elec_TJ'].values[0] / 3600 # TWh
    # calculating relative error
    reselec_err = (reselec_est - reselec_iea) / reselec_iea * 100 # %
    # displaying results
    print('Resdiential electricity demand validation for', ccode)
    print(f"estimated residential electricity demand: {reselec_est:,.3f} TWh")
    print(f"reported residential electricity demand: {reselec_iea:,.3f} TWh")
    print(f"estimation error: {reselec_err:.1f} %")

### 4.4 Saving base year electricity demand estimates

The purpose of this step is to save the re-distributed income and residential electricity demand estimates for the base year. These are saved as shapefiles.

In [ ]:
# saving base year estimates as shapefiles
for ccode in ccode_list:
    redist_name = ccode + '_redist_inc'

    locals()[redist_name].set_geometry('geometry', inplace=True, crs='EPSG:4326')
    
    locals()[redist_name].to_file('2_output/' + ccode + '_socioecon_elec_base.gpkg', driver='GPKG')

## End of base year validation script